# 02 - Data Preprocessing

This notebook demonstrates the data preprocessing pipeline for MMM.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Project root
PROJECT_ROOT = Path().absolute().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_data
from src.preprocessing import (
    apply_adstock,
    apply_saturation,
    add_calendar_features,
)

## 1. Load Raw Data

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "mmm_data.parquet"
df = load_data(DATA_PATH, currency="GBP")
df.head()

## 2. Adstock Transformation

Adstock models the carryover effect of advertising - how today's spend affects future sales.

In [ ]:
# Simulate spend data
spend = np.zeros(20)
spend[5] = 1000  # Spend pulse at week 5

# Apply different decay rates
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for alpha in [0.3, 0.5, 0.7]:
    adstocked = apply_adstock(spend, alpha, l_max=8)
    axes[0].plot(adstocked, label=f'alpha={alpha}')

axes[0].set_xlabel('Week')
axes[0].set_ylabel('Adstocked Spend')
axes[0].set_title('Adstock Effect (alpha = decay rate)')
axes[0].legend()

# Show decay visualization
weeks = np.arange(10)
for alpha in [0.3, 0.5, 0.7]:
    retention = alpha ** weeks
    axes[1].plot(weeks, retention, label=f'alpha={alpha}')

axes[1].set_xlabel('Weeks Since Spend')
axes[1].set_ylabel('Retention Rate')
axes[1].set_title('Carryover Effect Decay')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Saturation Transformation

Saturation models diminishing returns - more spend yields less incremental impact.

In [ ]:
spend_levels = np.linspace(0, 1000, 100)

fig, ax = plt.subplots(figsize=(8, 5))

for lam in [100, 300, 500]:
    saturated = apply_saturation(spend_levels, lam)
    ax.plot(spend_levels, saturated, label=f'lambda={lam}')

ax.set_xlabel('Spend')
ax.set_ylabel('Saturated Effect')
ax.set_title('Saturation Curve (logistic transformation)')
ax.legend()
plt.show()

## 4. Feature Engineering

Create control variables for the MMM.

In [ ]:
# Create weekly aggregation
df['date'] = pd.to_datetime(df['DATE_DAY'])
df['week'] = df['date'].dt.to_period('W').dt.start_time

df_weekly = df.groupby('week').agg({
    'ALL_PURCHASES_NET_PRICE': 'sum',
    'GOOGLE_PAID_SEARCH_SPEND': 'sum',
    'META_FACEBOOK_SPEND': 'sum',
}).reset_index()

# Add trend
df_weekly['trend'] = np.arange(len(df_weekly)) / len(df_weekly)

# Add seasonality
df_weekly['week_of_year'] = df_weekly['week'].dt.isocalendar().week / 52
df_weekly['month'] = df_weekly['week'].dt.month / 12

df_weekly.head(10)

## 5. Summary

Key preprocessing steps:
1. **Weekly aggregation** - Standard for MMM
2. **Adstock** - Captures carryover effects
3. **Saturation** - Models diminishing returns
4. **Control variables** - Trend, seasonality, holidays